In [1]:
import os


In [2]:
%pwd

'c:\\Users\\USER\\MLOPS-project\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\USER\\MLOPS-project'

In [5]:
import pandas as pd

In [6]:
data= pd.read_csv("artifacts/data_ingestion/winequality-red.csv")
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1599 entries, 0 to 1598
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1599 non-null   float64
 1   volatile acidity      1599 non-null   float64
 2   citric acid           1599 non-null   float64
 3   residual sugar        1599 non-null   float64
 4   chlorides             1599 non-null   float64
 5   free sulfur dioxide   1599 non-null   float64
 6   total sulfur dioxide  1599 non-null   float64
 7   density               1599 non-null   float64
 8   pH                    1599 non-null   float64
 9   sulphates             1599 non-null   float64
 10  alcohol               1599 non-null   float64
 11  quality               1599 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 150.0 KB


In [7]:
data.isnull().sum()

fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
dtype: int64

In [8]:
data.shape

(1599, 12)

In [9]:
data.columns

Index(['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
       'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density',
       'pH', 'sulphates', 'alcohol', 'quality'],
      dtype='object')

In [10]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataValidationConfig:
    root_dir: Path
    STATUS_FILE: str
    unzip_data_dir: Path
    all_schema: dict


In [11]:
from myproject.constants import *
from myproject.utils.common import read_yaml,create_directories

In [12]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath= CONFIG_FILE_PATH,
        params_filepath= PARAMS_FILE_PATH ,
        schema_filepath= SCHEMA_FILE_PATH):

        self.config=read_yaml(config_filepath)
        self.params=read_yaml(params_filepath)
        self.schema=read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])



    def get_data_validation_config(self)->DataValidationConfig:
        config=self.config.data_validation
        schema= self.schema.COLUMNS

        create_directories([config.root_dir])

        data_validation_config = DataValidationConfig(
            root_dir=config.root_dir,
            STATUS_FILE=config.STATUS_FILE,
            unzip_data_dir=config.unzip_data_dir,
            all_schema=schema,
        )

        return data_validation_config
    

In [13]:
import os
from myproject import logger

In [14]:
import pandas as pd
import os

class DataValidation:
    def __init__(self, config):
        self.config = config

    def validation_all_columns(self) -> bool:
        try:
            print("Reading file:", self.config.unzip_data_dir)
            print("Exists:", os.path.exists(self.config.unzip_data_dir))

            data = pd.read_csv(self.config.unzip_data_dir)

            all_cols = list(data.columns)
            all_schema = self.config.all_schema.keys()

            validation_status = True

            for col in all_cols:
                if col not in all_schema:
                    validation_status = False
                    break

            with open(self.config.STATUS_FILE, 'w') as f:
                f.write(f"Validation status: {validation_status}")

            return validation_status

        except Exception as e:
            raise e

In [17]:
try:
    config=ConfigurationManager()
    data_validation_config=config.get_data_validation_config()
    data_validation= DataValidation(config=data_validation_config)
    data_validation.validation_all_columns()
except Exception as e:
    raise e


[2026-04-26 23:25:40,268: INFO: common:yaml file: C:\Users\USER\MLOPS-project\config\config.yaml loaded successfully]
[2026-04-26 23:25:40,273: INFO: common:yaml file: param.yaml loaded successfully]
[2026-04-26 23:25:40,287: INFO: common:yaml file: schema.yaml loaded successfully]
[2026-04-26 23:25:40,289: INFO: common:created directory at: artifacts]
[2026-04-26 23:25:40,292: INFO: common:created directory at: artifacts/data_validation]
Reading file: artifacts/data_ingestion/winequality-red.csv
Exists: True
